<a href="https://colab.research.google.com/github/farankhandev/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farankhandev/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a Decision Tree Regressor for this assignment. My Week-4 baseline uses a rule-based action score based on CTR and average search position. For Week 5, I wanted to test whether other available content and performance features could predict that scoring pattern.

A Decision Tree is a good starting point because it can capture non-linear relationships and is easier to interpret than a more complex model. I limited the maximum depth to 5 to keep the model relatively simple and reduce unnecessary complexity.

The goal is not to claim that the model replaces or improves the Week-4 baseline. Instead, I am testing how well the selected features predict the baseline score on unseen data and which features the model relies on.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
df.head()

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
print("Rows:", len(df))

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))

print("\nKey signals:")
print(df[[
    "ctr",
    "avg_position",
    "engagement_rate",
    "impressions_90d"
]].describe())

Rows: 30000

Missing values:
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
cpc                   2468
dtype: int64

Key signals:
                ctr  avg_position  engagement_rate  impressions_90d
count  30000.000000   30000.00000     30000.000000     30000.000000
mean       0.510733      16.34238         2.534520      5200.366300
std        3.279162      15.21679         8.310096     16838.019547
min        0.000000       0.00000         0.000000         1.000000
25%        0.000000       6.20000         0.000000        81.000000
50%        0.070000      10.80000         0.000000       731.000000
75%        0.290000      22.30000         1.350000      3615.250000
max      100.000000     245.00000       100.000000    517715.000000


In [ ]:

df_model = df[df["impressions_90d"] >= 10].copy()

print("Original rows:", len(df))
print("Rows after filtering:", len(df_model))

Original rows: 30000
Rows after filtering: 26254


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a train/test split with 80% of the data for training and 20% for testing. I used the same filtered dataset as Week 4 (impressions_90d >= 10) so the comparison remains consistent.

The split is not grouped by client because the goal is to test whether the model can reproduce the Week-4 baseline scoring pattern on unseen content. The test set is kept separate from training so the model is evaluated on data it did not see during training.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_model,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Training rows: 21003
Testing rows: 5251


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I trained a Decision Tree Regressor using the training portion of the filtered dataset. The target is the Week-4 baseline score, and the model uses selected content and performance features to learn the scoring pattern.

I then evaluated the model on the held-out test set and compared its predictions with the Week-4 baseline scores. This comparison shows how closely the Decision Tree can reproduce the baseline scoring pattern on unseen data.

The goal is not to claim that the Decision Tree is better than the Week-4 baseline. It is a test of whether additional features can approximate the existing rule-based scoring approach. A useful improvement would require stronger validation and evidence that the model improves the actual content-optimization decisions.

In [ ]:
df_section3 = df_model.copy()

ctr_score = 1 - (
    (df_section3["ctr"] - df_section3["ctr"].min()) /
    (df_section3["ctr"].max() - df_section3["ctr"].min())
)

position_score = (
    (df_section3["avg_position"] - df_section3["avg_position"].min()) /
    (df_section3["avg_position"].max() - df_section3["avg_position"].min())
)

df_section3["baseline_score"] = (
    ctr_score * 50 +
    position_score * 50
)

print("Week-4 baseline score recreated.")
print(df_section3[[
    "content_id",
    "ctr",
    "avg_position",
    "baseline_score"
]].head())

Week-4 baseline score recreated.
             content_id   ctr  avg_position  baseline_score
0  content_304f48230142  0.76          10.6       54.140284
1  content_a1fb4e703a9e  0.05          20.3       60.212870
2  content_9aa793d4d895  0.09          36.5       68.362881
3  content_331d6c4de07b  0.49           6.2       52.347786
4  content_d99b7a2d90ca  0.13          44.0       72.101129


In [ ]:
from sklearn.model_selection import train_test_split

features = [
    "impressions_90d",
    "engagement_rate",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

X = df_section3[features].copy()
y = df_section3["baseline_score"].copy()


X = X.fillna(X.median())

print("Features:", X.shape)
print("Target:", y.shape)

Features: (26254, 10)
Target: (26254,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 21003
Testing rows: 5251


In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


In [ ]:
y_pred = model.predict(X_test)

print("Predictions created.")
print("First 10 predictions:")
print(y_pred[:10])

Predictions created.
First 10 predictions:
[57.44848228 58.95360851 57.44848228 60.62204218 59.99488239 52.94344867
 57.44848228 58.11405548 57.44848228 55.08430418]


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Decision Tree vs Week-4 baseline")
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²:", round(r2, 4))

Decision Tree vs Week-4 baseline
MAE: 5.0102
RMSE: 6.9601
R²: 0.2101


In [ ]:
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

feature_importance

,feature,importance
5,word_count,0.233904
7,content_age_days,0.223518
0,impressions_90d,0.176802
9,trend_pct,0.175549
8,days_since_last_update,0.078404
2,search_volume,0.051837
6,char_count,0.048405
4,cpc,0.007278
1,engagement_rate,0.004302
3,competition,0.000000


In [ ]:
comparison_table = pd.DataFrame({
    "metric": [
        "MAE",
        "RMSE",
        "R2"
    ],
    "value": [
        mae,
        rmse,
        r2
    ]
})

comparison_table

,metric,value
0,MAE,5.010213
1,RMSE,6.960146
2,R2,0.210085


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Decision Tree Regressor reproduced part of the Week-4 baseline scoring pattern on the unseen test data. The measured MAE was 5.01 and the RMSE was 6.96, while the R² was 0.21. This means the model captured some of the variation in the baseline score, but a substantial amount remained unexplained.

The model relied most on word_count (0.234), content_age_days (0.224), impressions_90d (0.177), and trend_pct (0.176). These feature-importance values show which features the tree used most when making its predictions, but they do not establish causation.

The model therefore should not be treated as a replacement for the Week-4 baseline. The results show that the selected features contain some information about the baseline scoring pattern, but the relatively low R² indicates that the Decision Tree does not reproduce the baseline particularly closely. Further feature selection, validation, and model comparison would be needed before using a predictive model for content-optimization decisions.

In [ ]:
error_analysis = pd.DataFrame({
    "actual_baseline_score": y_test,
    "predicted_baseline_score": y_pred
})

error_analysis["error"] = (
    error_analysis["actual_baseline_score"]
    - error_analysis["predicted_baseline_score"]
)

error_analysis["absolute_error"] = (
    error_analysis["error"].abs()
)

error_analysis = error_analysis.sort_values(
    "absolute_error",
    ascending=False
)

error_analysis.head(10)

,actual_baseline_score,predicted_baseline_score,error,absolute_error
10796,13.072518,57.448482,-44.375964,44.375964
18893,3.955375,46.628185,-42.672810,42.672810
29209,7.310076,46.628185,-39.318109,39.318109
15789,93.255578,58.941399,34.314179,34.314179
17287,91.277890,57.448482,33.829408,33.829408
13664,91.227181,57.448482,33.778698,33.778698
2824,92.089249,58.941399,33.147850,33.147850
10093,90.720081,58.114055,32.606026,32.606026
8719,91.886410,59.359013,32.527397,32.527397
2991,26.338350,58.114055,-31.775705,31.775705


In [ ]:
print("Mean error:", round(error_analysis["error"].mean(), 4))
print("Mean absolute error:", round(error_analysis["absolute_error"].mean(), 4))
print("Largest absolute error:", round(error_analysis["absolute_error"].max(), 4))

Mean error: -0.0796
Mean absolute error: 5.0102
Largest absolute error: 44.376


## Self-check


- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.